[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/WesamKhallaf73/pytorch-ground-up-notebooks/blob/main/notebooks/L04.ipynb)

# Lecture 04 — Creating, Inspecting & Indexing Tensors

*Make a tensor of any shape and type, inspect it in three questions, then reach in and pull out exactly the piece you need.*

**PyTorch from Ground Up · Volume I · Chapters 4–5** — companion notebook.

**How to use it:** run the cells from top to bottom. Before each code cell, *predict* what it will print — then run it and compare with the expected output (click ▸ to open it). Every exercise from the lecture is at the end.

By the end you will be able to:
- Create tensors with the eight factory functions, from your own data, and with the `_like` family.
- Inspect any tensor with three questions — shape, dtype, device — and cast it when it arrives wrong.
- Index by position, range, step, boolean mask and integer list.
- Know whether an index gave you a view or a copy — and when to `.clone()`.
- Write through an index, and recognise the pattern `torch.gather` solves.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)
print('PyTorch', torch.__version__)

## 01 · The factory gallery

Two skills you’ll use in every lecture after this: **making tensors** and **reaching into them**. Deliberately mechanical — they need to become reflexes so your attention can go to the interesting parts.

<sub>Source: lecture slides 3–6</sub>

🧪 *Interactive lab in the app:* **Eight factory functions** — try it there to see this idea move.

## 02 · Your own data and the _like family

Match, don’t hard-code: every factory has a **`_like` twin**. `zeros_like(x)` copies x’s shape and dtype — so your code keeps working when the batch size changes.

<sub>Source: lecture slides 7–10</sub>

**`from_data.py`** — predict the output, then run.

In [ ]:
# From a list
a = torch.tensor([1.0, 2.0, 3.0])
# From a nested list → 2-D
b = torch.tensor([[1, 2],
                  [3, 4]])
# From a NumPy array
import numpy as np
c = torch.from_numpy(np.array([5.0, 6.0]))
print(a.shape, b.shape, c.shape)

<details><summary>▸ Expected output</summary>

```
torch.Size([3]) torch.Size([2, 2]) torch.Size([2])
```
</details>

> ⚠️ **Watch out — tensor() copies; from_numpy() shares**  
> `torch.tensor()` always **copies** the data into a fresh, independent tensor. `torch.from_numpy()` **shares memory** with the original array — change one and the other changes too.
>
> Use `tensor()` when you want a clean independent copy; use `from_numpy()` when you want zero-cost interop and you understand the sharing.

**`like_family.py`** — predict the output, then run.

In [ ]:
x = torch.randn(3, 4)   # float32, shape (3,4)
z = torch.zeros_like(x) # same shape, same dtype
r = torch.randn_like(x) # random, same shape
print(z.shape, z.dtype)

<details><summary>▸ Expected output</summary>

```
torch.Size([3, 4])
torch.float32
```
</details>

## 03 · Three questions, every time

Most tensor bugs are a wrong answer to one of these three — caught late. **Ask early.**

<sub>Source: lecture slides 11–13</sub>

**`inspect.py`** — predict the output, then run.

In [ ]:
t = torch.randn(4, 8)
print(t.shape)     # how is it arranged?
print(t.dtype)     # what kind of numbers?
print(t.device)    # where does it live?

<details><summary>▸ Expected output</summary>

```
torch.Size([4, 8])
torch.float32
cpu
```
</details>

## 04 · Casting dtype and device

Tensors arrive in the wrong type, or the wrong place. **Casting fixes both.** Integer labels that need to be floats for a loss; a tensor stuck on the CPU while your model is on the GPU.

<sub>Source: lecture slides 14–18</sub>

**`cast_dtype.py`** — predict the output, then run.

In [ ]:
labels = torch.tensor([0, 1, 2])   # int64
print(labels.dtype)
floats = labels.float()            # → float32
print(floats.dtype)
half = labels.to(torch.float16)    # → float16
print(half.dtype)

<details><summary>▸ Expected output</summary>

```
torch.int64
torch.float32
torch.float16
```
</details>

**`cast_device.py`** — predict the output, then run.

In [ ]:
t = torch.randn(3)
print(t.device)                # cpu
device = "cuda" if torch.cuda.is_available() \
    else "cpu"
t2 = t.to(device)
print(t2.device)               # cuda:0 or cpu

<details><summary>▸ Expected output</summary>

```
cpu
cpu      (cuda:0 on a GPU machine)
```
</details>

> ⚠️ **Watch out — Casting returns a NEW tensor**  
> Both `.float()` and `.to(device)` return a new tensor — they do not change the original in place. You must capture the result: write `t = t.float()`, not `t.float()` on its own.
>
> A lonely `t.float()` with no assignment is a silent no-op — the most common casting bug there is.

**`create_typed.py`** — predict the output, then run.

In [ ]:
a = torch.zeros(3, 3,
    dtype=torch.float16, device="cpu")
print(a.dtype, a.device)

<details><summary>▸ Expected output</summary>

```
torch.float16 cpu
```
</details>

## 05 · Reaching inside: position, range, step

Now reach inside. One 4×5 matrix, used all the way through — so you can trace exactly what gets picked.

<sub>Source: lecture slides 20–26</sub>

**`matrix.py`** — predict the output, then run.

In [ ]:
m = torch.arange(20).reshape(4, 5)
print(m)

<details><summary>▸ Expected output</summary>

```
tensor([[ 0,  1,  2,  3,  4],
        [ 5,  6,  7,  8,  9],
        [10, 11, 12, 13, 14],
        [15, 16, 17, 18, 19]])
```
</details>

🧪 *Interactive lab in the app:* **Index the working matrix** — try it there to see this idea move.

> ⚠️ **Watch out — No negative steps**  
> The book prints a reversed matrix for `m[::-1]`, but real PyTorch raises `ValueError: step must be greater than zero` — negative-step slicing isn’t supported. Use `torch.flip(m, dims=[0])` instead.

## 06 · Boolean masks and fancy indexing

Say the **condition**, not the place. `m > 12` builds a True/False grid of the same shape; indexing with it returns a flat 1-D tensor of the matches. The grid shape is lost, because the number of matches can vary.

<sub>Source: lecture slides 27–31</sub>

**`boolean_mask.py`** — predict the output, then run.

In [ ]:
mask = m > 12
print(mask)
print(m[mask])

<details><summary>▸ Expected output</summary>

```
tensor([[False, False, False, False, False],
        [False, False, False, False, False],
        [False, False, False,  True,  True],
        [ True,  True,  True,  True,  True]])
tensor([13, 14, 15, 16, 17, 18, 19])
```
</details>

**`fancy_indexing.py`** — predict the output, then run.

In [ ]:
# Pick rows 0, 2, and 3
print(m[[0, 2, 3]])
# Pick specific (row, col) pairs
rows = torch.tensor([0, 1, 3])
cols = torch.tensor([4, 2, 0])
print(m[rows, cols])

<details><summary>▸ Expected output</summary>

```
tensor([[ 0,  1,  2,  3,  4],
        [10, 11, 12, 13, 14],
        [15, 16, 17, 18, 19]])
tensor([ 4,  7, 15])
```
</details>

## 07 · Views vs copies — the most important practical detail

Basic slicing hands you a **view**: the same memory, seen through a new window.

<sub>Source: lecture slides 32–36</sub>

🧪 *Interactive lab in the app:* **View or copy? Know before you write** — try it there to see this idea move.

**`write_through_index.py`** — predict the output, then run.

In [ ]:
m = torch.arange(20).reshape(4, 5)
m[0, :] = -1           # fill row 0 with -1
m[m < 0] = 0           # replace negatives
print(m)

<details><summary>▸ Expected output</summary>

```
tensor([[ 0,  0,  0,  0,  0],
        [ 5,  6,  7,  8,  9],
        [10, 11, 12, 13, 14],
        [15, 16, 17, 18, 19]])
```
</details>

## 08 · A quick note on gather

One more tool to recognise: `gather` — **from each row, pick the column named in another tensor**. You’ll meet it when picking each example’s correct-class score. Don’t master it now.

<sub>Source: lecture slides 37–38</sub>

**`gather.py`** — predict the output, then run.

In [ ]:
scores = torch.tensor([[0.1, 0.9, 0.3],
                       [0.8, 0.1, 0.4]])
labels = torch.tensor([[1], [0]])
picked = scores.gather(1, labels)
print(picked)

<details><summary>▸ Expected output</summary>

```
tensor([[0.9000],
        [0.8000]])
```
</details>

## The rule — worth remembering

**Making & inspecting**

1. `zeros`, `ones`, `rand`, `randn`, `arange`, `linspace`, `full`, `eye`, and `tensor()` cover almost every creation need.
2. The `_like` family clones shape and dtype from an existing tensor.
3. `torch.tensor()` copies data; `torch.from_numpy()` shares it.
4. Always inspect: `.shape`, `.dtype`, `.device`.
5. Cast with `.float()` / `.long()` / `.to(dtype)` / `.to(device)` — remember it returns a new tensor.
6. Set dtype and device at creation when you know them — cleaner than casting after.

**Reaching inside**

1. Single element: `t[row, col]`. Negative indices count from the end.
2. Slice: `t[start:stop:step]`. Colon `:` means “all.” Stop is excluded.
3. Boolean mask: `t[t > 0]` returns a flat copy of matching elements.
4. Fancy index: `t[[0, 2, 3]]` picks specific positions — also a copy.
5. Basic slicing returns a view (shared memory); boolean and fancy indexing return a copy.
6. Use `.clone()` when you need an independent copy from a view.

## Pause and try

Work each exercise in the empty cell below it. Open the worked answer only after you have tried.

### Exercise 01 · Make

Create one tensor with each of the eight factories and print its `.shape`/`.dtype`. Build `[10, 20, 30, 40, 50]` with both `arange` and `linspace` — identical? Check with `torch.equal`.

In [ ]:
# your code here


<details><summary>▸ Worked answer</summary>

`torch.arange(10, 51, 10)` gives int64 `[10, 20, 30, 40, 50]` (note the stop of 51 — the endpoint is excluded). `torch.linspace(10, 50, 5)` gives float32 `[10., 20., 30., 40., 50.]`. The values match but the dtypes differ, so compare after casting one side — a reminder to always ask the dtype question.
</details>

### Exercise 02 · Inspect & cast

Add `torch.tensor([1, 2, 3])` to `torch.tensor([1.0, 2.0, 3.0])` — what happens, and how do you fix it with casting?

In [ ]:
# your code here


<details><summary>▸ Worked answer</summary>

The first is int64, the second float32. Simple arithmetic like this is promoted to float (`tensor([2., 4., 6.])`), but many operations — matrix multiplies, losses — refuse mixed dtypes. The reliable habit is to cast explicitly: `a.float() + b`.
</details>

### Exercise 03 · Reach in

From `m = torch.arange(20).reshape(4, 5)`, pull the bottom-right 2×3 sub-block; build a boolean mask for the even numbers; take `s = m[0:2]`, set `s[0, 0] = -1` — does `m` change? Repeat with `.clone()`.

In [ ]:
# your code here


<details><summary>▸ Worked answer</summary>

`m[2:, 2:]` → `[[12, 13, 14], [17, 18, 19]]`. Even mask: `m % 2 == 0`. `s = m[0:2]` is a view, so `s[0, 0] = -1` changes `m[0, 0]` too. With `s = m[0:2].clone()`, `m` is untouched.
</details>

### Interview-style

You need a zeros tensor matching an input’s shape and dtype — what function, and why is it better than reading the shape by hand?

_Write your answer here before opening the model answer._

<details><summary>▸ Model answer</summary>

`torch.zeros_like(x)`. It copies shape, dtype and device from the reference in one call, so it stays correct when the batch size or type changes — nothing is hard-coded.
</details>

### Interview-style

Does `a = t[0:5]` share memory with `t`? What about `a = t[t > 0]`? When would you use `.clone()`?

_Write your answer here before opening the model answer._

<details><summary>▸ Model answer</summary>

`t[0:5]` is basic slicing, so it is a view that shares memory — writing to `a` writes to `t`. `t[t > 0]` is boolean indexing and returns an independent copy. Use `.clone()` whenever you have a view but need to modify it without touching the original.
</details>